# 📝 Step 4: 통합 파싱 규칙 기반 전체 회의록 텍스트(TSV) 추출 및 정제
본 노트북은 3단계에서 확정된 통합 파싱 규칙을 바탕으로 전체 광역의회 원본 회의록 JSON을 순회하여
발언자(성명, 직함, 소속구분), 안건명, 정제된 발언 본문을 추출하고 동일 화자의 연속 발언을 병합하여
표준 TSV(`cleaned_*.txt`) 데이터셋을 생성합니다.

### 📌 주요 전처리 기능
1. **의전/장내 괄호 멘트 제거**: `(개의)`, `(인사)`, `(박수)`, `(마이크 미사용)` 등 비정형 장내 소음 제거
2. **분절된 자간 복원 (`fix_fragmented_spacing`)**: `의 원 여 러 분` -> `의원 여러분` 등 한 글자씩 쪼개진 텍스트 자연스럽게 복원
3. **화자 정규화 (`parse_speaker`)**: 발언자 텍스트에서 직함(role), 성명(name), 소속구분(`의회` vs `집행부`) 분리
4. **연속 발언 병합 (Utterance Consolidation)**: 동일한 화자가 연속해서 말한 분할 문단들을 하나의 완성된 발언 단위로 결합

In [ ]:
import glob
import json
import os
import re
import time
from bisect import bisect_right
from pathlib import Path
from bs4 import BeautifulSoup, Tag
import pandas as pd

# ---------------------------------------------------------------------------
# 1. 환경 설정 및 경로 지정
# ---------------------------------------------------------------------------
TARGET_YEAR = "2024"

# 원본 JSON 경로 후보 자동 탐색
candidate_dirs = [
    Path(f"../../origin/{TARGET_YEAR}_광역 회의록"),
    Path(f"../origin/{TARGET_YEAR}_광역 회의록"),
    Path(f"./origin/{TARGET_YEAR}_광역 회의록"),
    Path(f"./data/org/{TARGET_YEAR}_광역 회의록"),
]
INPUT_DIR = str(next((p for p in candidate_dirs if p.exists()), Path(f"./origin/{TARGET_YEAR}_광역 회의록")))
OUTPUT_DIR = f"./data/to_text2/{TARGET_YEAR}_광역 회의록"
RULES_FILE = f"./data/to_text/llm_rules_outputs/consolidated_parsing_rules_{TARGET_YEAR}.json"

print(f"입력 원본 JSON 경로: {os.path.abspath(INPUT_DIR)}")
print(f"출력 정제 TSV 경로: {os.path.abspath(OUTPUT_DIR)}")
print(f"통합 규칙 파일: {os.path.abspath(RULES_FILE)}")

## 2. 동적 회의록 전처리기 클래스 (`DynamicCouncilPreprocessor`)

In [ ]:
class DynamicCouncilPreprocessor:
    """지방의회 HTML 회의록에서 안건, 발언자, 발언문을 추출하고 정제하는 클래스"""

    def __init__(self, min_char_len: int = 30):
        self.min_char_len = min_char_len
        self.formality_patterns = [
            r"\([^\)]*(?:개식|폐식|개의|산회|인사|기립|묵념|박수|마이크)[^\)]*\)",
        ]
        self.combined_regex = re.compile("|".join(self.formality_patterns), re.MULTILINE)

    def fix_fragmented_spacing(self, text: str) -> str:
        """1글자 단위로 쪼개진 텍스트('의 원 여 러 분')를 단어로 병합"""
        text = re.sub(r'\s+([,.\?!])', r'\1', text)
        tokens = text.split()
        fixed_tokens = []
        temp_word = ""
        for token in tokens:
            pure_text = re.sub(r'[^가-힣A-Za-z0-9]', '', token)
            if len(pure_text) <= 1:
                temp_word += token
            else:
                if temp_word:
                    fixed_tokens.append(temp_word)
                    temp_word = ""
                fixed_tokens.append(token)
        if temp_word:
            fixed_tokens.append(temp_word)
        return " ".join(fixed_tokens)

    def clean_text(self, text: str) -> str:
        """의전 괄호 제거 및 공백 정규화"""
        cleaned = self.combined_regex.sub(" ", text)
        cleaned = self.fix_fragmented_spacing(cleaned)
        return re.sub(r"\s+", " ", cleaned).strip()

    def parse_speaker(self, raw_speaker: str) -> tuple:
        """발언자 직함, 성명, 소속구분(의회/집행부) 분리"""
        text = re.sub(r"^[○□■\-\s]+", "", raw_speaker).strip()
        tokens = text.split()
        if len(tokens) >= 2:
            role, name = tokens[0], "".join(tokens[1:])
        elif len(tokens) == 1:
            role, name = "미상", tokens[0]
        else:
            return "미상", "미상", "기타"
        is_council = any(r in role for r in ["의원", "의장", "부의장", "위원장", "부위원장", "위원"])
        speaker_type = "의회" if is_council else "집행부"
        return role, name, speaker_type

    def extract_following_text(self, spk_tag: Tag, speaker_ids: set) -> str:
        """발언자 태그 뒤의 인접 텍스트를 다음 발언자 전까지 수집"""
        parts = []
        for sibling in spk_tag.next_siblings:
            if isinstance(sibling, Tag):
                if id(sibling) in speaker_ids or any(id(child) in speaker_ids for child in sibling.find_all(True)):
                    break
                parts.append(sibling.get_text(" ", strip=True))
            else:
                parts.append(str(sibling))
        return self.clean_text(" ".join(part for part in parts if part))

    def process_record(self, record: dict, rule: dict, source_record_no: int) -> list:
        """단일 회의록 JSON 레코드에서 발언 레코드 목록 추출"""
        results = []
        html_content = record.get("MINTS_HTML", "")
        if not html_content:
            return results

        soup = BeautifulSoup(html_content, "html.parser")
        for tag in soup(["script", "style", "form", "table", "ul.skipUl", "#topFunction"]):
            tag.decompose()

        agenda_selector = rule.get("agenda", "div")
        speaker_selector = rule.get("speaker", "span.spkrbld")
        utterance_selector = rule.get("utterance", "p")

        text_node_mode = (
            agenda_selector == "__TEXT_NODE_AGENDA__"
            and speaker_selector == "__TEXT_NODE_SPEAKER__"
        )

        if text_node_mode:
            speaker_tags = [t for t in soup.find_all("strong") if re.match(r"^\s*[○□■↘]\s*\S", t.get_text(" ", strip=True))]
            agenda_tags = [t for t in soup.find_all("strong") if re.match(r"^\s*\d+[.)．]\s*\S", t.get_text(" ", strip=True))]
            utterance_tags = []
            utterance_selector = ""
        else:
            try:
                speaker_tags = soup.select(speaker_selector)
                agenda_tags = soup.select(agenda_selector)
                utterance_tags = soup.select(utterance_selector) if utterance_selector else []
            except Exception:
                return results

        if not speaker_tags:
            speaker_tags = [t for t in soup.find_all("strong") if t.get_text(" ", strip=True).startswith(("○", "□", "■", "↘"))]
            utterance_selector = ""

        agenda_ids = {id(t) for t in agenda_tags}
        speaker_ids = {id(t) for t in speaker_tags}
        agenda_by_speaker = {}
        active_agenda = "일반안건"

        for node in soup.descendants:
            if not getattr(node, "name", None):
                continue
            if id(node) in agenda_ids:
                active_agenda = node.get_text(" ", strip=True)[:50] or "일반안건"
            if id(node) in speaker_ids:
                agenda_by_speaker[id(node)] = active_agenda

        for spk_tag in speaker_tags:
            raw_speaker = spk_tag.get_text(strip=True)
            role, name, speaker_type = self.parse_speaker(raw_speaker)
            agenda_title = agenda_by_speaker.get(id(spk_tag), "일반안건")
            utterance_cleaned = self.extract_following_text(spk_tag, speaker_ids)

            if len(utterance_cleaned) >= self.min_char_len:
                results.append({
                    "_source_record_no": source_record_no,
                    "date": record.get("MTG_DE", ""),
                    "assembly": record.get("RASMBLY_NM", ""),
                    "agenda_title": agenda_title,
                    "speaker_role": role,
                    "speaker_name": name,
                    "speaker_type": speaker_type,
                    "utterance": utterance_cleaned,
                })
        return results

## 3. 전체 회의록 전처리 및 TSV 변환 실행

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 통합 규칙 로드
parsing_rules = {}
if os.path.exists(RULES_FILE):
    with open(RULES_FILE, "r", encoding="utf-8") as f:
        parsing_rules = json.load(f)

json_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.json")))
if not json_files:
    json_files = sorted(glob.glob(os.path.join(INPUT_DIR, "**", "*.json"), recursive=True))

preprocessor = DynamicCouncilPreprocessor(min_char_len=30)
print(f"처리 대상 JSON 파일 수: {len(json_files)}개")

# 테스트로 처음 2개 파일 실행
for input_file in json_files[:2]:
    base_name = os.path.basename(input_file)
    file_key = os.path.splitext(base_name)[0]
    matched_rule = parsing_rules.get(file_key, {"agenda": "div", "speaker": "span.spkrbld", "utterance": "p"})
    
    print(f"\n[{base_name}] 전처리 시작... (규칙: {matched_rule})")
    try:
        with open(input_file, "r", encoding="utf-8") as f:
            raw_data = json.load(f)
        records = raw_data if isinstance(raw_data, list) else [raw_data]
        all_utterances = []
        for idx, rec in enumerate(records[:15], start=1):  # 테스트용 15건 레코드
            all_utterances.extend(preprocessor.process_record(rec, matched_rule, idx))
            
        df = pd.DataFrame(all_utterances)
        if not df.empty:
            # 연속 발언 병합
            condition = (
                (df['_source_record_no'] != df['_source_record_no'].shift()) |
                (df['speaker_name'] != df['speaker_name'].shift()) |
                (df['speaker_type'] != df['speaker_type'].shift()) |
                (df['agenda_title'] != df['agenda_title'].shift())
            )
            df['group_id'] = condition.cumsum()
            df_merged = df.groupby('group_id').agg({
                'date': 'first',
                'assembly': 'first',
                'agenda_title': 'first',
                'speaker_role': 'first',
                'speaker_name': 'first',
                'speaker_type': 'first',
                'utterance': lambda x: ' '.join(x)
            }).reset_index(drop=True)
            
            output_path = os.path.join(OUTPUT_DIR, f"cleaned_{file_key}.txt")
            df_merged.to_csv(output_path, sep="\t", index=False, encoding="utf-8-sig")
            print(f"✓ 추출 완료: 총 {len(df_merged)}건 발언 저장 -> {output_path}")
        else:
            print("⚠️ 추출된 발언 없음")
    except Exception as e:
        print(f"❌ 오류 발생: {e}")

## 4. 정제 결과 TSV 미리보기 및 품질 검사

In [ ]:
output_tsvs = glob.glob(os.path.join(OUTPUT_DIR, "cleaned_*.txt"))
if output_tsvs:
    sample_df = pd.read_csv(output_tsvs[0], sep="\t")
    print(f"파일: {output_tsvs[0]} (총 {len(sample_df)}행)")
    display(sample_df.head(5))
    print("\n[화자 소속 분포]")
    print(sample_df['speaker_type'].value_counts())
else:
    print("생성된 TSV 파일이 없습니다.")